# 01 — Create Benchmark Datasets: Delta vs Lance

**Purpose:** Generate a synthetic multimodal dataset **once** as in-memory Ray blocks, then write those same blocks to both **Delta** and **Lance**. This isolates the storage format as the variable for the throughput benchmark in `02_training_benchmark.ipynb`.

Covers stages **1–4** of [`README.md`](README.md): setup, generate, write, and the add-column ETL benchmark.

| | Delta (Parquet-backed) | Lance |
|---|---|---|
| Image storage | **`image_path`** ref → JPEG files in a Volume | **Inline** JPEG bytes (blob layout) |
| Writer | Ray → `write_databricks_table` (via SQL Warehouse) | Ray → `write_lance` (fragment + commit) |
| Add a column | `ALTER TABLE ADD COLUMN` + full backfill | `add_columns` — new column only, no rewrite |

The path-reference layout is how images are actually stored in Delta — it's the pattern the parent blueprint's failure-mode #1 is about (per-image GET hop). Lance stores bytes inline. That difference is the benchmark.

**Compute:** Databricks Classic Compute — **8 worker nodes × 16 CPUs** each.

---

**Outputs (per size tier):**
- Lance dataset at `/Volumes/{catalog}/{schema}/{volume}/synthetic_lance_{size}/`
- JPEG files at `/Volumes/{catalog}/{schema}/{volume}/synthetic_images_{size}/`
- Delta table `{catalog}.{schema}.synthetic_delta_{size}` (metadata + `image_path`)

**Next:** `02_training_benchmark.ipynb`

In [ ]:
# Install packages BEFORE setup_ray_cluster — installing after shuts the cluster down.
%pip install -qU "ray[data]==2.54.0" "lance==0.17.0" "pyarrow>=16.0" Pillow numpy pandas "databricks-sdk>=0.49.0"
dbutils.library.restartPython()

In [ ]:
# ── Widgets ───────────────────────────────────────────────────────────────
dbutils.widgets.dropdown("size", "10k", ["10k", "100k", "1m", "10m"], "Dataset size")
dbutils.widgets.text("catalog", "main", "UC catalog")
dbutils.widgets.text("schema", "ml_benchmark", "UC schema")
dbutils.widgets.text("volume", "lance_benchmark", "UC volume")
dbutils.widgets.text("warehouse_id", "", "SQL Warehouse ID (blank = provision)")
dbutils.widgets.text("seed", "42", "RNG seed")
dbutils.widgets.text("embedding_dim", "512", "Embedding dim")

size          = dbutils.widgets.get("size")
catalog       = dbutils.widgets.get("catalog")
schema        = dbutils.widgets.get("schema")
volume        = dbutils.widgets.get("volume")
SEED          = int(dbutils.widgets.get("seed"))
EMBEDDING_DIM = int(dbutils.widgets.get("embedding_dim"))

SIZE_MAP = {"10k": 10_000, "100k": 100_000, "1m": 1_000_000, "10m": 10_000_000}
N_ROWS   = SIZE_MAP[size]

# Fixed category set — MUST match 02_training_benchmark.ipynb.
CATEGORIES = ["cat", "dog", "car", "tree", "house", "flower", "boat", "bird"]

base_vol     = f"/Volumes/{catalog}/{schema}/{volume}"
lance_path   = f"{base_vol}/synthetic_lance_{size}"
images_dir   = f"{base_vol}/synthetic_images_{size}"      # JPEG files for the Delta branch
delta_table  = f"{catalog}.{schema}.synthetic_delta_{size}"

print(f"Size tier   : {size} ({N_ROWS:,} rows)")
print(f"Lance out   : {lance_path}")
print(f"Delta table : {delta_table}")
print(f"Delta images: {images_dir}")
print(f"Categories  : {CATEGORIES}")

In [ ]:
import os

# Credentials — set BEFORE setup_ray_cluster so Ray workers inherit them (Ray 2.41+).
os.environ["DATABRICKS_HOST"]  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

In [ ]:
# Ensure output + Ray tmp volumes exist, and the Delta-branch image dir.
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import catalog as sdk_catalog

w = WorkspaceClient()
for vol_name in [volume, "ray_tmp"]:
    try:
        w.volumes.read(f"{catalog}.{schema}.{vol_name}")
    except Exception:
        w.volumes.create(catalog_name=catalog, schema_name=schema, name=vol_name,
                         volume_type=sdk_catalog.VolumeType.MANAGED)
        print(f"Created volume {catalog}.{schema}.{vol_name}")

ray_tmp_path = f"/Volumes/{catalog}/{schema}/ray_tmp"
os.makedirs(images_dir, exist_ok=True)
print(f"Ray tmp     : {ray_tmp_path}")
print(f"Images dir  : {images_dir}")

## SQL Warehouse — provision or reuse

`ray.data.write_databricks_table` routes through a running SQL Warehouse. Provision-or-reuse by name so reruns don't spawn duplicates; serverless + short auto-stop keeps an idle warehouse from billing. Pin an existing one via the `warehouse_id` widget.

In [ ]:
from databricks.sdk.service.sql import State

WAREHOUSE_NAME = "ray-benchmark-warehouse"


def get_or_create_warehouse(warehouse_id="", name=WAREHOUSE_NAME,
                            cluster_size="Small", auto_stop_mins=10):
    if warehouse_id:
        return warehouse_id
    for wh in w.warehouses.list():
        if wh.name == name:
            if wh.state in (State.STOPPED, State.STOPPING):
                w.warehouses.start(wh.id).result()
            elif wh.state == State.STARTING:
                w.warehouses.get_and_wait(wh.id)
            print(f"Reusing warehouse '{name}' ({wh.id})")
            return wh.id
    created = w.warehouses.create(
        name=name, cluster_size=cluster_size, auto_stop_mins=auto_stop_mins,
        enable_serverless_compute=True, min_num_clusters=1, max_num_clusters=1,
    ).result()
    print(f"Created warehouse '{name}' ({created.id})")
    return created.id


warehouse_id = get_or_create_warehouse(dbutils.widgets.get("warehouse_id"))

In [ ]:
# Classic Compute Ray cluster — 8 worker nodes × 16 CPUs.
import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

try:
    shutdown_ray_cluster()
except Exception:
    pass

N_WORKER_NODES = 8
CPUS_PER_NODE  = 16

setup_ray_cluster(
    min_worker_nodes=N_WORKER_NODES,
    max_worker_nodes=N_WORKER_NODES,      # fixed size (min == max)
    num_cpus_worker_node=CPUS_PER_NODE,
    collect_log_to_path=ray_tmp_path,
)
ray.init(address="auto", ignore_reinit_error=True)

total_cpus = ray.cluster_resources().get("CPU", 0)
print(f"Total CPUs  : {total_cpus:.0f} | alive nodes: {sum(1 for n in ray.nodes() if n['Alive'])}")
assert total_cpus >= N_WORKER_NODES * CPUS_PER_NODE * 0.9, "Cluster did not fully start"

## Generate synthetic data (once)

`ray.data.range(N).map_batches(generate_batch)` fans generation across the cluster. Each row is seeded by `(SEED, id)`, so generation is deterministic and independent of block partitioning. The image is conditioned on category (hue) so the classification task in `02` is learnable; noise keeps the JPEG in the ~30–300KB range.

In [ ]:
import numpy as np


def _make_image(rng, category_idx, n_categories):
    """Procedural RGB image conditioned on category, JPEG-encoded to ~30-300KB."""
    import io
    from PIL import Image

    side = int(rng.integers(256, 512))
    base = np.zeros((side, side, 3), dtype=np.float32)
    hue = category_idx / n_categories
    base[..., 0] = 255 * hue
    base[..., 1] = 255 * (1 - hue)
    base[..., 2] = 128
    noise = rng.integers(0, 60, size=(side, side, 3))
    arr = np.clip(base + noise, 0, 255).astype(np.uint8)

    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, format="JPEG", quality=90)
    return buf.getvalue()


def generate_batch(batch, seed, categories, embedding_dim):
    ids = batch["id"]
    n_cat = len(categories)
    images, captions, embeddings, cats, brightness, quality = [], [], [], [], [], []
    for _id in ids:
        rng = np.random.default_rng([seed, int(_id)])
        cat_idx = int(rng.integers(0, n_cat))
        images.append(_make_image(rng, cat_idx, n_cat))
        captions.append(f"a photo of a {categories[cat_idx]} " + "x" * int(rng.integers(0, 40)))
        embeddings.append(rng.standard_normal(embedding_dim).astype(np.float32))
        cats.append(categories[cat_idx])
        brightness.append(float(rng.random()))
        quality.append(int(rng.integers(1, 6)))
    return {
        "id":         np.asarray(ids),
        "image":      np.asarray(images, dtype=object),
        "caption":    np.asarray(captions, dtype=object),
        "embedding":  np.asarray(embeddings, dtype=np.float32),
        "category":   np.asarray(cats, dtype=object),
        "brightness": np.asarray(brightness, dtype=np.float32),
        "quality":    np.asarray(quality, dtype=np.int32),
    }

In [ ]:
# Generate ONCE and materialize — both formats write from these same in-memory blocks.
override_blocks = max(64, N_ROWS // 5_000)

ds = (
    ray.data.range(N_ROWS, override_num_blocks=override_blocks)
    .map_batches(
        generate_batch,
        fn_kwargs={"seed": SEED, "categories": CATEGORIES, "embedding_dim": EMBEDDING_DIM},
        batch_size=512,
    )
    .materialize()
)
print(f"Generated {ds.count():,} rows")

total_image_bytes = ds.map_batches(
    lambda b: {"nbytes": np.array([sum(len(x) for x in b["image"])])},
    batch_size=512,
).sum("nbytes")
print(f"Raw image bytes: {total_image_bytes / 1e9:.3f} GB")

## Write — Lance (inline)

Each Ray write task emits an independent Lance fragment; a single driver-side commit merges fragment metadata into a new dataset version. Image bytes stored inline.

> **Note:** `ds.write_lance()` reports one combined write time. Splitting fragment-write vs commit time (per the README) needs the lower-level `lance.fragment.write_fragments()` + commit API — deferred here to keep the writer idiomatic.

In [ ]:
import time, os, lance


def dir_stats(path):
    total, nfiles = 0, 0
    for root, _, files in os.walk(path):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(root, f)); nfiles += 1
            except OSError:
                pass
    return total, nfiles


t0 = time.time()
ds.write_lance(lance_path)
lance_write_s = time.time() - t0

lds = lance.dataset(lance_path)
n_frag = len(lds.get_fragments())
lc_bytes, _ = dir_stats(lance_path)
print(f"Lance write   : {lance_write_s:6.2f}s | {N_ROWS / lance_write_s:>10,.0f} rows/s | "
      f"{lc_bytes / 1e6 / lance_write_s:6.1f} MB/s")
print(f"Lance on-disk : {lc_bytes / 1e9:.3f} GB across {n_frag} fragments")

## Write — Delta (path references)

The Databricks-native image pattern: JPEG bytes are written out as files in a Volume, and the Delta table holds an `image_path` string plus the metadata columns (no inline bytes). Writing files and the metadata table both fan out across Ray. The table is created via `ray.data.write_databricks_table` (SQL Warehouse).

In [ ]:
_images_dir = images_dir


def write_images_and_meta(batch):
    """Write each JPEG to the Volume; return the metadata row with image_path (no bytes)."""
    import os
    paths = []
    for _id, jpeg in zip(batch["id"], batch["image"]):
        p = os.path.join(_images_dir, f"{int(_id):012d}.jpg")
        with open(p, "wb") as f:
            f.write(jpeg)
        paths.append(p)
    return {
        "id":         batch["id"],
        "image_path": np.asarray(paths, dtype=object),
        "caption":    batch["caption"],
        "embedding":  batch["embedding"],
        "category":   batch["category"],
        "brightness": batch["brightness"],
        "quality":    batch["quality"],
    }


t0 = time.time()
meta_ds = ds.map_batches(write_images_and_meta, batch_size=512).materialize()
files_write_s = time.time() - t0
img_bytes, img_files = dir_stats(images_dir)
print(f"Delta JPEG files: {files_write_s:6.2f}s | {img_files:,} files | {img_bytes / 1e9:.3f} GB")

In [ ]:
# Write the metadata table to Delta via the SQL Warehouse.
# NOTE: verify write_databricks_table's signature against ray==2.54.0 on first run —
# arg names (table / catalog / schema / mode) can vary by Ray version.
t0 = time.time()
meta_ds.write_databricks_table(
    f"{catalog}.{schema}.synthetic_delta_{size}",
    warehouse_id=warehouse_id,
    mode="overwrite",
)
delta_write_s = time.time() - t0
delta_count = spark.sql(f"SELECT COUNT(*) AS n FROM {delta_table}").collect()[0]["n"]
print(f"Delta table   : {delta_write_s:6.2f}s | {delta_count:,} rows written")

In [ ]:
import pandas as pd

# Delta on-disk = metadata Parquet + the referenced JPEG files.
delta_meta_bytes = 0
try:
    detail = spark.sql(f"DESCRIBE DETAIL {delta_table}").collect()[0]
    delta_meta_bytes = detail["sizeInBytes"] or 0
except Exception:
    pass
delta_total_bytes = delta_meta_bytes + img_bytes

write_summary = pd.DataFrame([
    {"format": "lance", "write_s": round(lance_write_s, 2),
     "rows_per_s": round(N_ROWS / lance_write_s), "on_disk_GB": round(lc_bytes / 1e9, 3),
     "files": n_frag, "compression_x": round(total_image_bytes / lc_bytes, 2)},
    {"format": "delta", "write_s": round(files_write_s + delta_write_s, 2),
     "rows_per_s": round(N_ROWS / (files_write_s + delta_write_s)),
     "on_disk_GB": round(delta_total_bytes / 1e9, 3),
     "files": img_files + 1, "compression_x": round(total_image_bytes / max(1, img_bytes), 2)},
])
display(write_summary)

## Verify — round-trip + random access

Confirm the Delta path-referenced JPEG round-trips the same bytes Lance stored inline for the same `id`, and time Lance point lookups at start / middle / end — access cost should be roughly constant (O(1) fragment addressing), independent of row position.

In [ ]:
probe_ids = [0, N_ROWS // 2, N_ROWS - 1]

print("Lance random-access latency:")
lance_rows = {}
for pid in probe_ids:
    t0 = time.time()
    row = lds.take([pid], columns=["id", "image"]).to_pylist()[0]
    lance_rows[row["id"]] = row["image"]
    print(f"  take id={pid:>12,}: {(time.time() - t0) * 1000:6.2f} ms")

# Delta round-trip: read image_path, then GET the file (the per-image hop the benchmark measures)
pdf = spark.sql(
    f"SELECT id, image_path FROM {delta_table} WHERE id IN ({','.join(map(str, probe_ids))})"
).toPandas()
path_map = dict(zip(pdf["id"], pdf["image_path"]))

print("\nRound-trip (Delta file == Lance inline):")
for pid in probe_ids:
    with open(path_map[pid], "rb") as f:
        delta_bytes = f.read()
    ok = delta_bytes == lance_rows.get(pid)
    print(f"  id={pid:>12,}: {'OK' if ok else 'MISMATCH':>8}  ({len(delta_bytes) / 1024:.0f} KB)")

## ETL benchmark — backfill a new column

Compute a derived column once and add it to the existing dataset. Lance's `add_columns` writes only the new column; Delta must `ALTER TABLE ADD COLUMN` then backfill (rewrites the affected Parquet files). Derived column: the L2 norm of the embedding — a stand-in for any UDF-computed feature.

In [ ]:
import pyarrow as pa


def compute_norm(record_batch):
    """BatchUDF: receives a pyarrow.RecordBatch, returns the new column."""
    embs = np.stack(record_batch.column("embedding").to_pylist()).astype("float32")
    norms = np.linalg.norm(embs, axis=1).astype("float32")
    return pa.record_batch({"embedding_norm": pa.array(norms)})


# ── Lance: add_columns — no rewrite of existing data ───────────────────────
t0 = time.time()
lds.add_columns(compute_norm, read_columns=["embedding"])
lance_backfill_s = time.time() - t0
lc_bytes_after, _ = dir_stats(lance_path)
print(f"Lance add_columns : {lance_backfill_s:6.2f}s | +{(lc_bytes_after - lc_bytes) / 1e6:,.1f} MB (new column only)")

In [ ]:
# ── Delta: ALTER TABLE ADD COLUMN + backfill (rewrites affected files) ─────
from pyspark.sql import functions as F

t0 = time.time()
spark.sql(f"ALTER TABLE {delta_table} ADD COLUMN embedding_norm FLOAT")
# Embedding is stored as an array column; aggregate_norm via SQL higher-order function.
spark.sql(f"""
    UPDATE {delta_table}
    SET embedding_norm = SQRT(AGGREGATE(TRANSFORM(embedding, x -> x * x), CAST(0.0 AS DOUBLE), (acc, v) -> acc + v))
""")
delta_backfill_s = time.time() - t0
delta_meta_after = spark.sql(f"DESCRIBE DETAIL {delta_table}").collect()[0]["sizeInBytes"] or 0
print(f"Delta backfill    : {delta_backfill_s:6.2f}s | on-disk metadata now {delta_meta_after / 1e6:,.1f} MB")

In [ ]:
etl_summary = pd.DataFrame([
    {"format": "lance", "op": "add_columns", "wall_s": round(lance_backfill_s, 2),
     "MB_written": round((lc_bytes_after - lc_bytes) / 1e6, 1)},
    {"format": "delta", "op": "ALTER + backfill", "wall_s": round(delta_backfill_s, 2),
     "MB_written": round(delta_meta_after / 1e6, 1)},
])
display(etl_summary)

## Deferred write metrics

Left out of this draft — they need infra-level instrumentation:

- **Peak worker memory** during write — needs a per-worker memory sampler.
- **Object-store PUT count** — needs cloud provider request metrics (only meaningful on S3/GCS/ADLS, not a local Volume mount). Note the Delta branch issues one PUT *per image file* — a real small-file cost the inline Lance layout avoids.
- **Ray write-task concurrency** and **retry/error counts** — from the Ray dashboard / logs.

---

**Next:** `02_training_benchmark.ipynb` — read each format back through Ray Data + Ray Train and measure loading + training throughput.